# Table S5. Interrater reliability and model agreement (225-excerpt holdout)

Reproduces Supplementary Table S5 of the PICU-Aware manuscript.

**Inputs** (place in `DATA_DIR`):

| File | Contents |
|---|---|
| `coder1_codes.json`, `coder2_codes.json` | R1's and R2's independent, pre-adjudication labels |
| `consensus_codes_holdout.json` | Reconciled consensus reference standard |
| `model_codes_holdout.json` | Raw LLM output for the holdout, with confidence ratings |
| `Final_Codebook.xlsx` | Codebook; maps each of the 29 substantive codes to its domain |

**Method.** Each excerpt is scored at the excerpt-by-code (present/absent) level against the codes of each domain (A: 8, B: 10, C: 11). Codes are placed in their codebook domain regardless of the column in which a coder entered them. The administrative background code (PARTICIPANT_BACKGROUND) is excluded. Model codes are restricted to medium and high confidence. Blank consensus cells are true negatives. The pooled row is the micro-average of per-domain cells. 95% CIs are percentile bootstrap intervals (3000 resamples over excerpts, which preserves within-excerpt dependence among codes).

In [ ]:
DATA_DIR = "data" # input file location
B = 3000 # bootstrap resamples
SEED = 20250101 # fixed seed for reproducible CIs
OUT_CSV = "table_s5.csv"


## Functions

In [2]:
import json, re, random
from collections import defaultdict
import openpyxl

DOMS = ["A", "B", "C"]
DOMAIN_NAMES = {
    "A": "A: Situation awareness and team coordination",
    "B": "B: Usability and workflow integration",
    "C": "C: Explainability and trust",
}
SYNONYMS = {
    "EXPLANATION_PLAUSABILITY": "EXPLANATION_PLAUSIBILITY",
    "FUTURE_FEATURE": "FUTURE_FEATURES",
    "CALIBRATION_PREF": "CALIB_PREF",
    "CALIBRATION_UNCERTAINTY": "CALIB_PREF",
    "APPRPRIATE_RELIANCE": "APPROPRIATE_RELIANCE",
    "ADOPTION_FRICTION": "ADOPTION",
}
IGNORE = {"", "NO_CODE", "(none)", "None", "PARTICIPANT_BACKGROUND", "DEMOGRAPHICS"}


def load_codebook(path):
    """Return {code: domain} for the 29 substantive codes (administrative background code excluded)."""
    ws = openpyxl.load_workbook(path, read_only=True)["Codes"]
    cb, dom = {}, None
    for r in list(ws.iter_rows(values_only=True))[1:]:
        cells = [str(c).strip() for c in r[:3] if c]
        if r[0] and str(r[0])[:2] in ("A.", "B.", "C.", "D."):
            dom = str(r[0])[0]
        codes = [c for c in cells if re.fullmatch(r"[A-Z_]+", c)]
        if codes and dom in DOMS and codes[0] not in IGNORE:
            cb[codes[0]] = dom
    return cb


def normalize(token):
    t = token.strip()
    if re.fullmatch(r"[A-Za-z]+( [A-Za-z]+)+", t):          # e.g. 'TEMPORAL REASONING'
        t = t.upper().replace(" ", "_")
    return SYNONYMS.get(t, t)


def to_sets(tokens, cb, unknown):
    """Map raw code tokens to {domain: set(codes)}, placing each code in its codebook (home) domain."""
    out = {d: set() for d in DOMS}
    for tok in tokens:
        c = normalize(tok)
        if c in IGNORE:
            continue
        if c not in cb:
            unknown.add(c)
            continue
        out[cb[c]].add(c)
    return out


def split(v):
    return re.split(r"[,;\n]+", str(v)) if v else []


def load_human(path, cb, unknown):
    """Coder or consensus file: one string column per domain."""
    out = {}
    for r in json.load(open(path)):
        toks = []
        for k, v in r.items():
            if k != "excerpt_id" and "Demograph" not in k:
                toks += split(v)
        out[r["excerpt_id"]] = to_sets(toks, cb, unknown)
    return out


def load_model(path, cb, unknown, confidence=("high", "medium")):
    """Model file: list of {code, confidence, ...} per domain; keep only the given confidence levels."""
    out = {}
    for r in json.load(open(path)):
        toks = [x["code"] for k in ("domainA_codes", "domainB_codes", "domainC_codes")
                for x in (r.get(k) or []) if x.get("confidence") in confidence]
        out[r["excerpt_id"]] = to_sets(toks, cb, unknown)
    return out


def cells(ids, X, Y, universe, doms):
    """Excerpt-by-code 2x2 counts; X = test/rater 1, Y = reference/rater 2."""
    tp = fp = fn = tn = 0
    for e in ids:
        for d in doms:
            x, y = X[e][d], Y[e][d]
            a, b, c = len(x & y), len(x - y), len(y - x)
            tp += a; fp += b; fn += c; tn += len(universe[d]) - a - b - c
    return tp, fp, fn, tn


def metrics(tp, fp, fn, tn):
    n = tp + fp + fn + tn
    po = (tp + tn) / n
    px, py = (tp + fp) / n, (tp + fn) / n
    pe_k = px * py + (1 - px) * (1 - py)
    pi = (px + py) / 2
    pe_g = 2 * pi * (1 - pi)
    prec = tp / (tp + fp) if tp + fp else float("nan")
    rec = tp / (tp + fn) if tp + fn else float("nan")
    return {
        "agreement_pct": 100 * po,
        "kappa": (po - pe_k) / (1 - pe_k),
        "ac1": (po - pe_g) / (1 - pe_g),
        "precision": prec,
        "recall": rec,
        "f1": 2 * tp / (2 * tp + fp + fn),
        "pct_model_codes_absent_from_reference": 100 * (1 - prec),   # 1 - precision
        "pct_reference_codes_missed": 100 * (1 - rec),               # 1 - recall
    }


def bootstrap(ids, X, Y, universe, doms, keys, B=3000, seed=20250101):
    """Percentile 95% CIs, resampling excerpts (keeps within-excerpt dependence among codes)."""
    rng = random.Random(seed)
    ids = list(ids)
    draws = defaultdict(list)
    for _ in range(B):
        s = [ids[rng.randrange(len(ids))] for _ in ids]
        m = metrics(*cells(s, X, Y, universe, doms))
        for k in keys:
            draws[k].append(m[k])
    ci = {}
    for k, v in draws.items():
        v.sort()
        ci[k] = (v[int(0.025 * B)], v[int(0.975 * B) - 1])
    return ci

### Load data

In [3]:
import os
cb = load_codebook(os.path.join(DATA_DIR, "Final_Codebook.xlsx"))
universe = {d: {c for c, dom in cb.items() if dom == d} for d in DOMS}
print("Substantive codes per domain:", {d: len(universe[d]) for d in DOMS})

unknown = set()
coder1 = load_human(os.path.join(DATA_DIR, "coder1_codes.json"), cb, unknown)
coder2 = load_human(os.path.join(DATA_DIR, "coder2_codes.json"), cb, unknown)
consensus = load_human(os.path.join(DATA_DIR, "consensus_codes_holdout.json"), cb, unknown)
model = load_model(os.path.join(DATA_DIR, "model_codes_holdout.json"), cb, unknown)

ids = sorted(consensus)
assert set(ids) == set(coder1) == set(coder2) == set(model), "excerpt IDs differ across files"
print("Holdout excerpts:", len(ids))
print("Unrecognized code labels (should be empty):", unknown or "none")

Substantive codes per domain: {'A': 8, 'B': 10, 'C': 11}
Holdout excerpts: 225
Unrecognized code labels (should be empty): none


### Compute  metrics table (S5)

In [4]:
import pandas as pd

rows = []
for scope in DOMS + ["Pooled"]:
    doms = DOMS if scope == "Pooled" else [scope]
    h = metrics(*cells(ids, coder1, coder2, universe, doms))
    m = metrics(*cells(ids, model, consensus, universe, doms))
    h_ci = bootstrap(ids, coder1, coder2, universe, doms, ["kappa", "ac1"], B=B, seed=SEED)
    m_ci = bootstrap(ids, model, consensus, universe, doms, ["precision", "recall", "f1", "kappa", "ac1"], B=B, seed=SEED)
    rows.append({
        "Domain": DOMAIN_NAMES.get(scope, "Overall (pooled)"),
        "Human agreement, %": h["agreement_pct"],
        "Human κ": h["kappa"], "Human κ CI": h_ci["kappa"],
        "Human AC1": h["ac1"], "Human AC1 CI": h_ci["ac1"],
        "Model precision": m["precision"], "Model recall": m["recall"],
        "Model F1": m["f1"], "Model F1 CI": m_ci["f1"],
        "Model κ": m["kappa"], "Model κ CI": m_ci["kappa"],
        "Model AC1": m["ac1"], "Model AC1 CI": m_ci["ac1"],
        "Model codes absent from reference, % (1 − precision)": m["pct_model_codes_absent_from_reference"],
        "Reference codes missed by model, % (1 − recall)": m["pct_reference_codes_missed"],
        "Absent CI, %": tuple(100 * (1 - x) for x in reversed(m_ci["precision"])),
        "Missed CI, %": tuple(100 * (1 - x) for x in reversed(m_ci["recall"])),
    })
raw = pd.DataFrame(rows)

### Metrics table final (S5)

In [5]:
f2 = lambda x: f"{x:.2f}"
ci = lambda t: f"({t[0]:.2f}, {t[1]:.2f})"
table = pd.DataFrame({
    "Domain": raw["Domain"],
    "Agreement, %": raw["Human agreement, %"].map(lambda x: f"{x:.1f}"),
    "κ (95% CI)": [f"{f2(a)} {ci(b)}" for a, b in zip(raw["Human κ"], raw["Human κ CI"])],
    "AC1 (95% CI)": [f"{f2(a)} {ci(b)}" for a, b in zip(raw["Human AC1"], raw["Human AC1 CI"])],
    "Precision": raw["Model precision"].map(f2),
    "Recall": raw["Model recall"].map(f2),
    "F1-score (95% CI)": [f"{f2(a)} {ci(b)}" for a, b in zip(raw["Model F1"], raw["Model F1 CI"])],
    "Model κ (95% CI)": [f"{f2(a)} {ci(b)}" for a, b in zip(raw["Model κ"], raw["Model κ CI"])],
    "Model AC1 (95% CI)": [f"{f2(a)} {ci(b)}" for a, b in zip(raw["Model AC1"], raw["Model AC1 CI"])],
})
table.to_csv(OUT_CSV, index=False)
pd.set_option("display.max_colwidth", None)
table

,Domain,"Agreement, %",κ (95% CI),AC1 (95% CI),Precision,Recall,F1-score (95% CI),Model κ (95% CI),Model AC1 (95% CI)
0,A: Situation awareness and team coordination,87.4,"0.27 (0.21, 0.33)","0.85 (0.83, 0.87)",0.62,0.56,"0.59 (0.52, 0.65)","0.55 (0.48, 0.61)","0.91 (0.89, 0.93)"
1,B: Usability and workflow integration,87.8,"0.39 (0.33, 0.44)","0.85 (0.83, 0.86)",0.73,0.68,"0.71 (0.66, 0.75)","0.67 (0.62, 0.72)","0.93 (0.91, 0.94)"
2,C: Explainability and trust,91.5,"0.36 (0.29, 0.43)","0.90 (0.89, 0.92)",0.75,0.60,"0.66 (0.60, 0.72)","0.64 (0.58, 0.70)","0.95 (0.94, 0.96)"
3,Overall (pooled),89.1,"0.35 (0.31, 0.38)","0.87 (0.86, 0.88)",0.70,0.62,"0.66 (0.63, 0.69)","0.63 (0.60, 0.66)","0.93 (0.92, 0.94)"


### Error rates

In [6]:
err = pd.DataFrame({
    "Domain": raw["Domain"],
    "Model codes absent from reference, % (95% CI)": [f"{a:.0f} ({c[0]:.0f}–{c[1]:.0f})" for a, c in
        zip(raw["Model codes absent from reference, % (1 − precision)"], raw["Absent CI, %"])],
    "Reference codes missed by model, % (95% CI)": [f"{a:.0f} ({c[0]:.0f}–{c[1]:.0f})" for a, c in
        zip(raw["Reference codes missed by model, % (1 − recall)"], raw["Missed CI, %"])],
})
err.to_csv("table_s5_error_rates.csv", index=False)
err

,Domain,"Model codes absent from reference, % (95% CI)","Reference codes missed by model, % (95% CI)"
0,A: Situation awareness and team coordination,38 (31–46),44 (38–51)
1,B: Usability and workflow integration,27 (22–33),32 (26–36)
2,C: Explainability and trust,25 (19–32),40 (33–47)
3,Overall (pooled),30 (26–33),38 (35–41)
